# 실습 2b · 민감정보 · 내부 정보 마스킹 후 분석 (OWASP LLM02)

외부 LLM에 보내기 전에 가릴 것: 개인정보(주민번호 · 카드 · 계좌 · 휴대폰 · 이름), 우리 서버 IP 대역
그대로 둘 것: 공격 출발지 IP (탐지 · 차단에 필요한 침해 지표, 고객 개인정보가 아님)

순서: 규칙 확인 → 단순 마스킹 → 가명 마스킹 → 보내기 전 재검사 → 같은 프롬프트로 분석 비교

## 1) 가릴 값의 규칙 (정규식)

In [1]:
import re

PII_RULES = {                                       # 이름: (정규식, 바꿀 표시)
    "주민번호": (r"\d{6}-[1-4]\d{6}", "[RRN]"),
    "카드":    (r"\b\d{4}-\d{4}-\d{4}-\d{4}\b", "[CARD]"),
    "계좌":    (r"\b\d{3}-\d{3}-\d{6}\b", "[ACCT]"),
    "휴대폰":  (r"\b01[016789]-\d{3,4}-\d{4}\b", "[PHONE]"),
    "이름":    (r"(?<=name=)[가-힣]{2,4}", "[NAME]"),
}
INTERNAL_IP = r"\b149\.171\.126\.\d{1,3}\b"   # 우리 서버 대역 (UNSW-NB15 실험망)

def find_pii(text):
    """가려야 할 값의 종류별 건수"""
    found = {name: len(re.findall(p, text)) for name, (p, _) in PII_RULES.items() if re.search(p, text)}
    if re.search(INTERNAL_IP, text):
        found["내부 서버 IP"] = len(re.findall(INTERNAL_IP, text))
    return found

def mask_pii(line):
    for pattern, tag in PII_RULES.values():
        line = re.sub(pattern, tag, line)   #텍스트(line) 안에서 정규식 pattern에 일치하는 개인정보를 찾아 지정된 tag로 바꿔칩니다.
    return line

### 규칙 확인: 교육용 가상 고객정보 2줄

In [2]:
PII_DEMO = [
    "2026-09-25 09:03:46 [WEB] src=203.0.113.25 uri=/ib/login.do status=200 name=홍길동 phone=010-1234-5678",
    "2026-09-25 09:05:10 [TXN] name=홍길동 rrn=900101-1234567 acct=110-234-567890 card=9410-1234-5678-9012",
]
for line in PII_DEMO:
    print(find_pii(line), "\n  →", mask_pii(line))

{'휴대폰': 1, '이름': 1} 
  → 2026-09-25 09:03:46 [WEB] src=203.0.113.25 uri=/ib/login.do status=200 name=[NAME] phone=[PHONE]
{'주민번호': 1, '카드': 1, '계좌': 1, '이름': 1} 
  → 2026-09-25 09:05:10 [TXN] name=[NAME] rrn=[RRN] acct=[ACCT] card=[CARD]


## 2) UNSW-NB15 로그 6줄 (sample_logs.txt 2 · 8~12번째 줄)

In [3]:
LINES = open("sample_logs.txt", encoding="utf-8").read().splitlines()
RAW = [LINES[i - 1] for i in [2, 8, 9, 10, 11, 12]]
for line in RAW:
    print(find_pii(line), line[28:110])

{'내부 서버 IP': 1} src=59.166.0.3:46973 dst=149.171.126.0:80 proto=tcp service=http state=FIN dur=9.1
{'내부 서버 IP': 1} src=175.45.176.1:20549 dst=149.171.126.10:111 proto=udp service=- state=INT dur=0.
{'내부 서버 IP': 1} src=175.45.176.1:61099 dst=149.171.126.17:111 proto=tcp service=- state=FIN dur=0.
{'내부 서버 IP': 1} src=175.45.176.1:38267 dst=149.171.126.10:80 proto=tcp service=http state=FIN dur=
{'내부 서버 IP': 1} src=175.45.176.1:30074 dst=149.171.126.18:80 proto=tcp service=http state=FIN dur=
{'내부 서버 IP': 1} src=175.45.176.1:63277 dst=149.171.126.18:110 proto=tcp service=pop3 state=FIN dur


## 3) 단순 마스킹: IP를 모두 [IP]로

In [4]:
ANY_IP = r"\b\d{1,3}(?:\.\d{1,3}){3}\b"
simple = [re.sub(ANY_IP, "[IP]", mask_pii(line)) for line in RAW]
for line in simple:
    print(line[28:110])

src=[IP]:46973 dst=[IP]:80 proto=tcp service=http state=FIN dur=9.108 spkts=8 dpkt
src=[IP]:20549 dst=[IP]:111 proto=udp service=- state=INT dur=0.000 spkts=2 dpkts=
src=[IP]:61099 dst=[IP]:111 proto=tcp service=- state=FIN dur=0.633 spkts=10 dpkts
src=[IP]:38267 dst=[IP]:80 proto=tcp service=http state=FIN dur=0.410 spkts=10 dpk
src=[IP]:30074 dst=[IP]:80 proto=tcp service=http state=FIN dur=0.248 spkts=10 dpk
src=[IP]:63277 dst=[IP]:110 proto=tcp service=pop3 state=FIN dur=1.272 spkts=34 dp


## 4) 가명 마스킹: 같은 서버 = 같은 가명, 공격 출발지는 유지
가명표는 원래 IP로 되돌리는 열쇠이므로 PC 밖으로 보내지 않습니다.

In [5]:
def mask_logs(lines):
    table = {}                                          # 원래 IP → 가명
    for line in lines:
        for ip in re.findall(INTERNAL_IP, line):
            table.setdefault(ip, f"SRV_{len(table) + 1:03d}")
    masked = [re.sub(INTERNAL_IP, lambda m: table[m.group(0)], mask_pii(l)) for l in lines]
    return masked, table

masked, table = mask_logs(RAW)
for line in masked:
    print(line[28:110])
print("\n가명표 (PC 안에만 보관):", table)

src=59.166.0.3:46973 dst=SRV_001:80 proto=tcp service=http state=FIN dur=9.108 spk
src=175.45.176.1:20549 dst=SRV_002:111 proto=udp service=- state=INT dur=0.000 spk
src=175.45.176.1:61099 dst=SRV_003:111 proto=tcp service=- state=FIN dur=0.633 spk
src=175.45.176.1:38267 dst=SRV_002:80 proto=tcp service=http state=FIN dur=0.410 s
src=175.45.176.1:30074 dst=SRV_004:80 proto=tcp service=http state=FIN dur=0.248 s
src=175.45.176.1:63277 dst=SRV_004:110 proto=tcp service=pop3 state=FIN dur=1.272 

가명표 (PC 안에만 보관): {'149.171.126.0': 'SRV_001', '149.171.126.10': 'SRV_002', '149.171.126.17': 'SRV_003', '149.171.126.18': 'SRV_004'}


## 5) 보내기 전 재검사: 남아 있으면 멈춤 (fail-closed)

In [6]:
def assert_clean(lines):
    left = find_pii("\n".join(lines))
    if left:
        raise ValueError(f"마스킹 누락 {left} → 전송 중단")

assert_clean(simple)
assert_clean(masked)
print("단순 · 가명 마스킹 모두 통과 → 전송 가능")

try:
    assert_clean(RAW[:1])                   # 원본을 넣으면 막히는지 확인
except ValueError as e:
    print("원본은:", e)

단순 · 가명 마스킹 모두 통과 → 전송 가능
원본은: 마스킹 누락 {'내부 서버 IP': 1} → 전송 중단


## 6) 같은 프롬프트로 분석 비교

In [7]:
from openai import OpenAI

ENV = "mono"          # "ollama" = 내 PC,  "mono" = MonoRouter(원격)
if ENV == "mono":
    client = OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1",
                    api_key="mr_openai_e0di9bdw3iFdUDG1hgAxEfo_tH_DXI2z")
    MODEL = "gpt-5.5"
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:3b"
print(ENV, MODEL)

mono gpt-5.5


In [8]:
def ask(prompt, system=None, temperature=0.2, show=True):
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    limit = {"max_tokens": 500} if ENV == "ollama" else {}   # 작은 모델이 끝없이 길게 답하는 것을 막음
    stream = client.chat.completions.create(model=MODEL, messages=messages,
                                            temperature=temperature, stream=True, **limit)
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [9]:
PROMPT = """[역할] 당신은 금융기관 SOC 분석가입니다.
[맥락] 한 줄 = 네트워크 연결 1건. state=INT는 응답 없이 끝난 연결. SRV_001 같은 값은 가명 처리한 우리 서버(같은 번호 = 같은 서버), [IP]는 가린 IP.
판단 기준: 한 출발지가 여러 서버의 같은 포트를 두드린 뒤 서비스에 접속하면 정찰 → 침투 시도 의심
[제약] 로그에 있는 내용만 근거로 쓴다. 가린 값을 추측해 복원하지 않는다.
[출력 형식] JSON으로만 답한다.
{"verdict": "위협 / 정상 / 판단 보류", "same_source_flow": 같은 출발지가 여러 서버를 탐색한 뒤 접속한 흐름이 보이면 true 아니면 false, "evidence": ["근거"]}
[로그]
"""
for name, lines in [("단순 마스킹", simple), ("가명 마스킹", masked)]:
    print(f"\n===== {name} =====")
    ask(PROMPT + "\n".join(lines))


===== 단순 마스킹 =====
{"verdict":"판단 보류","same_source_flow":false,"evidence":["로그의 src와 dst가 모두 [IP]로 가려져 있어 같은 출발지가 여러 서버를 대상으로 했는지 확인할 수 없다.","동일 출발지가 여러 서버의 같은 포트를 두드린 뒤 서비스에 접속한 흐름은 로그상 식별되지 않는다.","state=INT인 연결은 2015-01-22 11:50:49Z UDP/111 1건이나, 이후 동일 출발지의 침투 시도 흐름으로 연결할 근거가 없다."]}

===== 가명 마스킹 =====
{
  "verdict": "위협",
  "same_source_flow": true,
  "evidence": [
    "동일 출발지 175.45.176.1이 2015-01-22 11:50:49Z에 SRV_002:111/udp로 state=INT 연결을 시도했으며 응답 dpkts=0으로 종료됨",
    "동일 출발지 175.45.176.1이 2015-01-22 11:50:52Z에 SRV_003:111/tcp로 연결하여 여러 서버의 동일 포트 111을 접촉함",
    "이후 동일 출발지 175.45.176.1이 2015-01-22 11:51:31Z에 SRV_002:80 http, 11:52:18Z에 SRV_004:80 http, 11:52:21Z에 SRV_004:110 pop3 서비스에 state=FIN으로 접속함",
    "한 출발지가 여러 서버의 같은 포트 111을 두드린 뒤 http/pop3 서비스 접속 흐름이 확인되어 정찰 후 침투 시도 의심 기준에 부합함"
  ]
}


### 관찰 포인트
- 단순 마스킹은 출발지와 서버가 모두 [IP]가 되어 '한 곳이 여러 서버를 탐색한 뒤 접속한 흐름'이 사라집니다. same_source_flow · verdict를 비교합니다.
- 가명 마스킹은 우리 서버 구조는 숨기면서 '같은 서버'라는 관계는 남깁니다.
- 규칙은 정해진 형식만 잡습니다. '담당자 홍길동 승인' 같은 줄은 놓칩니다 → 외부 LLM에는 꼭 필요한 필드만 골라 보내는 것이 우선입니다.